# 04 — Backtest Results

The study's **authoritative results**. Everything below is read from
`reports/results/metrics.csv` and `run_manifest.json`, which `make run` produced from
`configs/pairs.yaml` — so nothing here can quietly disagree with what the pipeline
actually computed. The figures are regenerated live through
`pairs_teardown.study.run_study`, the same function the script calls.

This notebook deliberately contains **no strategy logic**. Where you see a hedge ratio or
a Sharpe, it came out of the package.

Two comparisons are mandatory in this project and structure everything that follows:

| Principle | What it demands | Where |
|---|---|---|
| 4 — gross vs net | every result stated before *and* after transaction costs | §3, §4 |
| 5 — IS vs OOS | parameters fit in-sample; out-of-sample scored once, reported as-is | §2, §5 |

> **Not this notebook:** `03_backtest_explore.ipynb` is earlier exploratory work that fit
> the hedge ratio on the full sample and never split IS/OOS. Its numbers are diagnostics,
> not findings.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pairs_teardown.config import load_config
from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.plotting.charts import plot_drawdown, plot_equity_curve
from pairs_teardown.study import run_study

# Notebooks run from notebooks/; every path below is anchored to the repo root so the
# notebook behaves identically however the kernel was launched.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

cfg = load_config(ROOT / "configs" / "pairs.yaml")
metrics = pd.read_csv(ROOT / "reports" / "results" / "metrics.csv")
manifest = json.loads((ROOT / "reports" / "results" / "run_manifest.json").read_text())

OFFICIAL = [p.name for p in cfg.official_pairs]
SANITY = [p.name for p in cfg.sanity_pairs]

pd.set_option("display.precision", 4)
print(f"metrics.csv: {len(metrics)} rows covering {metrics.pair.nunique()} pairs")
print(f"official (pre-specified): {OFFICIAL}")
print(f"sanity check (added later): {SANITY}")


## 1. Run provenance

What was run, with which parameters. `window`, `entry` and `exit` are **pre-registered
and frozen** — chosen a priori (60 days ≈ a trading quarter; 2.0/0.5 are the textbook
bands), never selected by looking at a return. §6 shows what tuning them would have
been worth, which is precisely why they are frozen.


In [ ]:
frozen = ["window", "entry", "exit", "signal_hedge", "sizing_hedge"]
costs = ["commission_bps", "slippage_bps"]
sample = ["data_start", "in_sample_end", "data_end"]

for label, keys in [("FROZEN SIGNAL", frozen), ("COSTS", costs), ("SAMPLE", sample)]:
    print(f"{label}")
    for k in keys:
        print(f"  {k:<16} {manifest[k]}")

print("\nStatic SIZING hedge ratios (fit on in-sample data ONLY, then frozen):")
for name, g in manifest["hedge_ratios"].items():
    tag = "official" if name in OFFICIAL else "sanity"
    print(f"  {name:<10} g = {g:7.4f}   [{tag}]")

print(f"\nrun at {manifest['run_utc']}")


## 2. Headline — the three pre-specified pairs

Net total return, in-sample versus out-of-sample. This is the study's actual claim; the
sanity-check pairs are held back to §5 and labelled there.


In [ ]:
def table(period_basis_value, pairs, value):
    """Pivot the long-form metrics into pair x period for one basis."""
    v = metrics[
        (metrics.basis == period_basis_value)
        & (metrics.period != "full")
        & (metrics.pair.isin(pairs))
    ]
    out = v.pivot_table(index="pair", columns="period", values=value)
    return out.reindex(pairs)[["in_sample", "out_of_sample"]]


headline = pd.concat(
    {
        "total return %": table("net", OFFICIAL, "total_return") * 100,
        "annualized %": table("net", OFFICIAL, "annualized_return") * 100,
        "Sharpe": table("net", OFFICIAL, "sharpe"),
    },
    axis=1,
)
headline.round(2)


**Every pre-specified pair loses money after costs, in both periods** — six of six
cells negative. The out-of-sample column was scored once, with every parameter frozen,
and is reported exactly as it came out.

This is the expected result of the teardown, not a bug to hunt. A pairs strategy on
economically-linked large caps, traded on a 60-day z-score at 6 bps per side, does not
clear its own transaction costs.


## 3. Gross versus net (principle 4)

The comparison that most published pairs-trading backtests omit. `drag` is the
annualized return the strategy hands to the broker.


In [ ]:
rows = []
for period in ["in_sample", "out_of_sample"]:
    for pair in OFFICIAL:
        sel = metrics[(metrics.pair == pair) & (metrics.period == period)]
        g = sel[sel.basis == "gross"].iloc[0]
        n = sel[sel.basis == "net"].iloc[0]
        rows.append(
            {
                "pair": pair,
                "period": period,
                "gross ann %": g.annualized_return * 100,
                "net ann %": n.annualized_return * 100,
                "drag %": (g.annualized_return - n.annualized_return) * 100,
                "gross Sharpe": g.sharpe,
                "net Sharpe": n.sharpe,
                "trades": int(g.n_trades),
            }
        )

gross_net = pd.DataFrame(rows).set_index(["period", "pair"])
gross_net.round(2)


Read the Sharpe columns rather than the returns, because they show how badly the
comparison misleads when only the gross number is published:

- **SPY/VOO in-sample** goes from a *positive* gross Sharpe to roughly **−1.3** net. It is
  the same trades, priced honestly.
- **SPY/VOO out-of-sample** is the extreme case: a gross Sharpe near zero becomes about
  **−2.4** net. Because the two ETFs track the same index, the spread is almost perfectly
  tight — the gross edge is tiny, its volatility is tinier, and a near-constant cost
  charge divided by a very small standard deviation produces a large negative ratio.
  A near-degenerate pair does not have a small cost problem; it has the worst one.
- **FOXA/FOX out-of-sample** is gross-flat (+0.01%/yr) and net-negative. Costs alone
  decide the sign.

The pattern is consistent: the gross series hover around zero, and costs push all of them
under. Nothing here had an edge that costs merely *reduced*.


## 4. Where the money goes

If the cost story above is real, the drag should be predictable from first principles
rather than being an unexplained residual. One unit of spread is long \$1 of A and short
\$g of B, so a unit change in position trades `(1 + |g|)` of notional and is charged
commission + slippage on it:

$$\text{annual drag} \;\approx\; \underbrace{\text{turnover}}_{\text{one-way, annualized}} \times\; (1 + |g|) \;\times\; \text{(commission + slippage)}$$

Predicted from the reported turnover and hedge ratio, versus the drag actually observed:


In [ ]:
bps = (manifest["commission_bps"] + manifest["slippage_bps"]) / 1e4

rows = []
for period in ["in_sample", "out_of_sample"]:
    for pair in OFFICIAL:
        sel = metrics[(metrics.pair == pair) & (metrics.period == period)]
        g = sel[sel.basis == "gross"].iloc[0]
        n = sel[sel.basis == "net"].iloc[0]
        actual = g.annualized_return - n.annualized_return
        predicted = g.turnover * (1 + abs(g.hedge_ratio)) * bps
        rows.append(
            {
                "pair": pair,
                "period": period,
                "turnover": g.turnover,
                "1+|g|": 1 + abs(g.hedge_ratio),
                "predicted drag %": predicted * 100,
                "actual drag %": actual * 100,
                "actual / predicted": actual / predicted,
            }
        )

recon = pd.DataFrame(rows).set_index(["period", "pair"])
recon.round(3)


The ratio sits between **0.966 and 0.998** in all six cells. The cost model is doing
exactly what its docstring says and nothing else — no hidden charge, no missing one.

That closes off the most common escape route from this kind of result ("the cost
assumption must be too harsh"). The assumption is 1 bp commission + 5 bps slippage per
side. It is not aggressive for liquid US large caps, and the drag it produces
(0.6–1.1%/yr) is fully explained by how often the strategy trades. To make these pairs
profitable you would need the *gross* edge to be larger, and §3 shows it is approximately
zero.


## 5. Full metrics, and the sanity-check pairs

The three pre-specified pairs in full, then the three added later. The second group is
reported **separately and labelled** so that adding it can never be read as — or quietly
become — a search over tickers.


In [ ]:
cols = [
    "total_return",
    "annualized_return",
    "sharpe",
    "max_drawdown",
    "turnover",
    "hit_rate",
    "n_trades",
    "n_periods",
]


def full_table(pairs):
    v = metrics[(metrics.pair.isin(pairs)) & (metrics.period != "full")]
    out = v.set_index(["pair", "period", "basis"])[cols].sort_index()
    return out.reindex(
        pd.MultiIndex.from_product(
            [pairs, ["in_sample", "out_of_sample"], ["gross", "net"]],
            names=["pair", "period", "basis"],
        )
    )


print("PRE-SPECIFIED (the study's claim)")
display(full_table(OFFICIAL).round(4))


In [ ]:
print("SANITY CHECK — added after the fact, reported separately, not part of the claim")
display(full_table(SANITY).round(4))


Three things to be honest about in this table.

**The only profitable pair is one that was not pre-specified.** KO/PEP returns +14.1% net
out-of-sample while all three official pairs lose. That is the data-snooping problem in a
single row: had KO/PEP been in the pre-registered set and the others discovered later, the
identical numbers would read as a success. Pre-registration is what stops the story being
rewritten around whichever pair happened to work, and it is why KO/PEP is quarantined in
this table rather than promoted into §2.

**Hit rates are ~50% everywhere** (0.47–0.55, computed over active days only). The
strategy is not winning slightly more often than it loses; it is a coin flip that pays a
toll on every flip.

**FOXA/FOX has 709 in-sample days against 1,763 for the others.** FOX only began trading
on 2019-03-13, after the Disney transaction. Its in-sample window is ~2.8 years, so its
hedge ratio is fit on less than half the data the others get and every one of its
statistics is correspondingly noisier. This is a limitation of the pair, not something to
correct for.


## 6. Equity curves and drawdowns

Regenerated live from `pairs_teardown.study.run_study` — the same code path `make run`
uses, so these cannot drift from the tables above. The dashed line marks the IS/OOS
boundary: everything to its right was scored once, with the hedge ratio and all
parameters already frozen.

On each equity chart the **gap between the gross and net lines is the transaction-cost
wedge** — the mechanism of §4, drawn.


In [ ]:
prices_raw = load_or_download(
    list(cfg.tickers), cfg.data.start, cfg.data.end, ROOT / cfg.data.cache_dir
)
runs = {r.pair.name: r for r in run_study(prices_raw, cfg, official_only=True)}
split_date = pd.Timestamp(cfg.split.in_sample_end)

for name in OFFICIAL:
    r = runs[name]
    gross_equity = (1 + r.result.gross_returns).cumprod()

    fig = plot_equity_curve(
        r.result.equity_curve, gross_equity, title=f"{name} — equity (gross vs net)"
    )
    fig.axes[0].axvline(split_date, ls="--", lw=1, color="0.4")
    fig.axes[0].annotate(
        " OOS →", (split_date, fig.axes[0].get_ylim()[1]), va="top", fontsize=9, color="0.3"
    )
    plt.show()

    fig = plot_drawdown(r.result.equity_curve, title=f"{name} — net drawdown")
    fig.axes[0].axvline(split_date, ls="--", lw=1, color="0.4")
    plt.show()


The curves grind downward rather than collapsing. That is the signature of a cost
problem rather than a blown-up position: a small, near-constant toll compounding over
2,515 trading days, not a handful of bad trades. The drawdown charts show no single
catastrophic event on the official pairs — the strategy is not being killed by tail risk,
it is being killed by friction.


## 7. What this notebook does and does not establish

**Establishes.** With the window, entry and exit thresholds fixed a priori, the sizing
hedge ratio fit on in-sample data only, and 6 bps per side of cost, all three
pre-specified pairs are net-negative in-sample and out-of-sample; and the cost drag that
produces this is quantitatively accounted for (§4).

**Does not establish.** That pairs trading cannot work. This is one parameterization, six
tickers, one cost assumption and one decade. Specifically not tested: intraday or
higher-frequency execution, a larger and genuinely pre-registered universe, dynamic
position sizing, or a cost model that reflects a market maker's fills rather than a
retail taker's.

**Not corrected for.** Survivorship — all six tickers were selected in 2025 from companies
that still exist and still trade. A pair that de-listed or was acquired mid-sample never
entered the universe, which biases the results *upward*. Since the finding is negative
even with that bias, correcting it would only strengthen the conclusion. See
`05_writeup.ipynb` §2 for the full argument.
